# Validate Mart Susceptibility



Validate final susceptibility reporting mart

Grain:
source release + cohort + standardized organism
+ reviewed antibiotic

The mart uses first eligible culture-organisms selected
independently of antibiotic testing or susceptibility result.

Categorized denominator:
Susceptible + Resistant + Intermediate

Other states remain explicit:
Inconclusive, Synergism, missing, conflict, untested.

1. Confirm mart grain is unique.
Expected result: zero rows.

In [1]:
%sql
select
    source_version,
    cohort_id,
    organism_standardized,
    antibiotic,
    count(*) as row_count

from workspace.micro_dev.mart_susceptibility

group by
    source_version,
    cohort_id,
    organism_standardized,
    antibiotic

having count(*) > 1;

,source_version,cohort_id,organism_standardized,antibiotic,row_count


2. Confirm every eligible record is either tested or untested.
Expected result: zero rows.

In [2]:
%sql
select
    cohort_id,
    organism_standardized,
    antibiotic,
    eligible_count,
    tested_endpoint_count,
    untested_count

from workspace.micro_dev.mart_susceptibility

where tested_endpoint_count + untested_count <> eligible_count;

,cohort_id,organism_standardized,antibiotic,eligible_count,tested_endpoint_count,untested_count


3. Confirm tested endpoints reconcile across result states.
Expected result: zero rows.

Categorized =
Susceptible + Resistant + Intermediate

In [3]:
%sql
select
    cohort_id,
    organism_standardized,
    antibiotic,

    tested_endpoint_count,

    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count,

    inconclusive_count,
    synergism_count,
    missing_result_count,
    conflict_result_count

from workspace.micro_dev.mart_susceptibility

where tested_endpoint_count <>
      categorized_count
    + inconclusive_count
    + synergism_count
    + missing_result_count
    + conflict_result_count

   or categorized_count <>
      susceptible_count
    + resistant_count
    + intermediate_count;

,cohort_id,organism_standardized,antibiotic,tested_endpoint_count,categorized_count,susceptible_count,resistant_count,intermediate_count,inconclusive_count,synergism_count,missing_result_count,conflict_result_count


4. Confirm analytical counts cannot exceed denominators.
Expected result: zero rows.

In [4]:
%sql
select
    cohort_id,
    organism_standardized,
    antibiotic,
    eligible_count,
    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count

from workspace.micro_dev.mart_susceptibility

where categorized_count > eligible_count
   or susceptible_count > categorized_count
   or resistant_count > categorized_count
   or intermediate_count > categorized_count;

,cohort_id,organism_standardized,antibiotic,eligible_count,categorized_count,susceptible_count,resistant_count,intermediate_count


5. Confirm every cohort-organism has every reviewed antibiotic
defined for that organism.

Expected result: zero rows.

In [5]:
%sql
with expected_pair_counts as (

    select
        organism_standardized,
        count(*) as expected_antibiotic_count

    from workspace.micro_dev.reportable_organism_antibiotic_pairs

    where include_primary_dashboard = 1
      and review_status = 'reviewed'

    group by organism_standardized

),

actual_pair_counts as (

    select
        cohort_id,
        organism_standardized,
        count(distinct antibiotic) as actual_antibiotic_count

    from workspace.micro_dev.mart_susceptibility

    group by
        cohort_id,
        organism_standardized

)

select
    a.cohort_id,
    a.organism_standardized,
    e.expected_antibiotic_count,
    a.actual_antibiotic_count

from actual_pair_counts a

inner join expected_pair_counts e
    on a.organism_standardized = e.organism_standardized

where a.actual_antibiotic_count <> e.expected_antibiotic_count;

,cohort_id,organism_standardized,expected_antibiotic_count,actual_antibiotic_count


6. Confirm eligible_count is constant across antibiotics for
each cohort-organism and matches int_first_culture_organism.

Expected result: zero rows.

In [6]:
%sql
with source_counts as (

    select
        source_version,
        cohort_id,
        organism_standardized,
        count(*) as expected_eligible_count

    from workspace.micro_dev.int_first_culture_organism

    group by
        source_version,
        cohort_id,
        organism_standardized

),

mart_counts as (

    select
        source_version,
        cohort_id,
        organism_standardized,
        min(eligible_count) as min_eligible_count,
        max(eligible_count) as max_eligible_count

    from workspace.micro_dev.mart_susceptibility

    group by
        source_version,
        cohort_id,
        organism_standardized

)

select
    m.source_version,
    m.cohort_id,
    m.organism_standardized,
    s.expected_eligible_count,
    m.min_eligible_count,
    m.max_eligible_count

from mart_counts m

inner join source_counts s
    on m.source_version = s.source_version
   and m.cohort_id = s.cohort_id
   and m.organism_standardized = s.organism_standardized

where m.min_eligible_count <> m.max_eligible_count
   or m.min_eligible_count <> s.expected_eligible_count;

,source_version,cohort_id,organism_standardized,expected_eligible_count,min_eligible_count,max_eligible_count


7. Confirm display-status rule.

Project policy:
categorized_count >= 30 -> display
categorized_count < 30  -> insufficient_data

Expected result: zero rows.

In [7]:
%sql
select
    cohort_id,
    organism_standardized,
    antibiotic,
    categorized_count,
    display_status

from workspace.micro_dev.mart_susceptibility

where (
        categorized_count >= 30
        and display_status <> 'display'
      )
   or (
        categorized_count < 30
        and display_status <> 'insufficient_data'
      );

,cohort_id,organism_standardized,antibiotic,categorized_count,display_status


8. Inspect primary blood results.

These are the values that will eventually feed the first
Tableau susceptibility view.

In [8]:
%sql
select
    organism_standardized,
    antibiotic,

    eligible_count,
    tested_endpoint_count,
    untested_count,
    categorized_count,

    susceptible_count,
    resistant_count,
    intermediate_count,

    inconclusive_count,
    synergism_count,
    missing_result_count,
    conflict_result_count,

    categorized_result_coverage_pct,
    susceptible_pct,
    resistant_pct,
    display_status

from workspace.micro_dev.mart_susceptibility

where cohort_id = 'blood'

order by
    organism_standardized,
    pair_order;

,organism_standardized,antibiotic,eligible_count,tested_endpoint_count,untested_count,categorized_count,susceptible_count,resistant_count,intermediate_count,inconclusive_count,synergism_count,missing_result_count,conflict_result_count,categorized_result_coverage_pct,susceptible_pct,resistant_pct,display_status
0,Enterobacter cloacae complex,Gentamicin,243,243,0,243,238,5,0,0,0,0,0,100.0,97.9,2.1,display
1,Enterobacter cloacae complex,Meropenem,243,243,0,243,242,1,0,0,0,0,0,100.0,99.6,0.4,display
2,Enterobacter cloacae complex,Trimethoprim/Sulfamethoxazole,243,243,0,243,225,18,0,0,0,0,0,100.0,92.6,7.4,display
3,Enterobacter cloacae complex,Ciprofloxacin,243,243,0,242,230,9,3,0,0,0,1,99.6,95.0,3.7,display
4,Enterobacter cloacae complex,Ceftriaxone,243,243,0,226,185,38,3,17,0,0,0,93.0,81.9,16.8,display
5,Enterobacter cloacae complex,Piperacillin/Tazobactam,243,243,0,225,201,17,7,17,0,0,1,92.6,89.3,7.6,display
6,Enterococcus faecalis,Ampicillin,530,530,0,530,530,0,0,0,0,0,0,100.0,100.0,0.0,display
7,Enterococcus faecalis,Vancomycin,530,530,0,530,524,6,0,0,0,0,0,100.0,98.9,1.1,display
8,Enterococcus faecalis,Penicillin,530,518,12,517,516,1,0,0,0,0,1,97.5,99.8,0.2,display
9,Enterococcus faecalis,Linezolid,530,320,210,319,315,1,3,0,0,1,0,60.2,98.7,0.3,display


9. Trace one important endpoint:
blood E. coli / ceftriaxone.

In [9]:
%sql
select *

from workspace.micro_dev.mart_susceptibility

where cohort_id = 'blood'
  and organism_standardized = 'Escherichia coli'
  and antibiotic = 'Ceftriaxone';

,susceptibility_mart_key,source_version,cohort_id,organism_standardized,antibiotic,pair_order,eligible_count,tested_endpoint_count,categorized_count,susceptible_count,resistant_count,intermediate_count,inconclusive_count,synergism_count,missing_result_count,conflict_result_count,untested_count,categorized_result_coverage_pct,susceptible_pct,resistant_pct,display_status
0,f7b934ebd7f09a61968a1a0765fdd4bd,2025-04-11,blood,Escherichia coli,Ceftriaxone,1,3021,3020,3015,2542,464,9,0,0,1,4,1,99.8,84.3,15.4,display
